# Deformable DETR Training on Colab

This notebook trains the repository's Deformable DETR implementation with:

- checkpoints stored on Google Drive
- automatic resume from the latest checkpoint
- TensorBoard metrics instead of Weights & Biases
- configurable Waymo segment count and Drive checkpoint folder

Set `CHECKPOINT_DIR` to an empty string to train from scratch without saving checkpoints.

In [ ]:
from google.colab import auth, drive

auth.authenticate_user()
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/yharidy/object_detection.git /content/object_detection
%cd /content/object_detection
!pip install -q -r requirements.txt
!pip install -q -e .

## Dataset partitions

This notebook uses Waymo's official partitions:

- `training`: used to update model weights.
- `validation`: used after each epoch to monitor generalization and select the best checkpoint or hyperparameters.
- `testing`: held back for the final evaluation after model and hyperparameters are frozen.

Set `TRAIN_NUM_SEGMENTS`, `VAL_NUM_SEGMENTS`, and `TEST_NUM_SEGMENTS` in the configuration cell to limit loading while debugging. `-1` means all available segments.

`CHECKPOINT_DIR` should point to a folder inside mounted Google Drive. Set it to `''` to disable checkpoint loading and saving.

In [ ]:
import torch
from pathlib import Path

DATA_ROOT = 'gs://waymo_open_dataset_v_2_0_1'
N_SEGMENTS = -1
TRAIN_SPLIT = 'training'
VAL_SPLIT = 'validation'
TEST_SPLIT = 'testing'
TRAIN_NUM_SEGMENTS = N_SEGMENTS
VAL_NUM_SEGMENTS = N_SEGMENTS
TEST_NUM_SEGMENTS = N_SEGMENTS
CHECKPOINT_DIR = '/content/drive/MyDrive/deformable_detr/checkpoints/v1'
RESUME = True

BATCH_SIZE = 1
NUM_WORKERS = 0
TARGET_SIZE = (320, 320)
NUM_EPOCHS = 2
NUM_CLASSES = 5
NUM_QUERIES = 100
NUM_LEVELS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
NUM_HEADS = 8
HIDDEN_DIM = 256
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
CHECKPOINT_EVERY_STEPS = 100
PREDICTIONS_EVERY_STEPS = 50
PREDICTION_SCORE_THRESHOLD = 0.5
MAX_PREDICTION_IMAGES = 4
USE_AMP = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = USE_AMP and DEVICE.type == 'cuda'

print('Device:', DEVICE)
print('Train split:', TRAIN_SPLIT, 'segments:', TRAIN_NUM_SEGMENTS)
print('Validation split:', VAL_SPLIT, 'segments:', VAL_NUM_SEGMENTS)
print('Test split:', TEST_SPLIT, 'segments:', TEST_NUM_SEGMENTS)
print('Checkpoint directory:', CHECKPOINT_DIR or 'disabled')
print('TensorBoard directory:', str(Path(CHECKPOINT_DIR) / 'tensorboard') if CHECKPOINT_DIR else '/content/tensorboard/deformable_detr')
print('Resume enabled:', RESUME)
print('Automatic mixed precision:', USE_AMP)
print('Prediction logging interval:', PREDICTIONS_EVERY_STEPS)

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from src.datasets.camera_2d.collate import camera_2d_collate_fn
from src.datasets.camera_2d.dataset import Camera2DDataset
from src.datasets.camera_2d.transforms import Camera2DTransform
from src.domain.enums import CameraPosition
from src.models.detr.deformable_detr import DeformableDetr
from src.models.detr.detr_loss import DETRLoss
from src.sources.waymo.factory import build_waymo_loaders
from src.utils.metrics import evaluate_detr_model
from src.utils.box_utils import cxcywh_to_xyxy

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

cameras = [CameraPosition.FRONT]
transform = Camera2DTransform(target_image_size=TARGET_SIZE)

train_loaders = build_waymo_loaders(
    data_root=DATA_ROOT,
    split=TRAIN_SPLIT,
    cameras=cameras,
    load_camera_labels=True,
    num_segments=TRAIN_NUM_SEGMENTS,
)
val_loaders = build_waymo_loaders(
    data_root=DATA_ROOT,
    split=VAL_SPLIT,
    cameras=cameras,
    load_camera_labels=True,
    num_segments=VAL_NUM_SEGMENTS,
)

train_dataset = Camera2DDataset(
    loaders=train_loaders,
    cameras=cameras,
    transform=transform,
)
val_dataset = Camera2DDataset(
    loaders=val_loaders,
    cameras=cameras,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=camera_2d_collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=camera_2d_collate_fn,
)

print('Training segments:', len(train_loaders))
print('Training samples:', len(train_dataset))
print('Training batches per epoch:', len(train_loader))
print('Validation segments:', len(val_loaders))
print('Validation samples:', len(val_dataset))
print('Validation batches:', len(val_loader))
print('Test split is reserved for final evaluation after model selection:', TEST_SPLIT)

In [ ]:
model = DeformableDetr(
    num_levels=NUM_LEVELS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    num_classes=NUM_CLASSES,
    num_queries=NUM_QUERIES,
    num_heads=NUM_HEADS,
    hidden_dim=HIDDEN_DIM,
    pretrained_backbone=True,
).to(DEVICE)

criterion = DETRLoss(
    image_width=TARGET_SIZE[0],
    image_height=TARGET_SIZE[1],
    num_classes=NUM_CLASSES,
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

checkpoint_path = Path(CHECKPOINT_DIR) / 'latest.pt' if CHECKPOINT_DIR else None
log_dir = Path(CHECKPOINT_DIR) / 'tensorboard' if CHECKPOINT_DIR else Path('/content/tensorboard/deformable_detr')
if CHECKPOINT_DIR:
    Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(log_dir))

start_epoch = 0
start_batch = 0
global_step = 0

if RESUME and checkpoint_path is not None and checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    if checkpoint.get('scaler_state') is not None:
        scaler.load_state_dict(checkpoint['scaler_state'])
    start_epoch = checkpoint['next_epoch']
    start_batch = checkpoint['next_batch']
    global_step = checkpoint['global_step']
    print(f'Resumed from epoch {start_epoch}, batch {start_batch}, step {global_step}')
else:
    print('Training from scratch')

In [ ]:
def save_checkpoint(next_epoch, next_batch, global_step):
    if checkpoint_path is None:
        return
    state = {
        'next_epoch': next_epoch,
        'next_batch': next_batch,
        'global_step': global_step,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict() if USE_AMP else None,
        'config': {
            'num_classes': NUM_CLASSES,
            'num_queries': NUM_QUERIES,
            'target_size': TARGET_SIZE,
            'hidden_dim': HIDDEN_DIM,
            'checkpoint_dir': CHECKPOINT_DIR,
        },
    }
    temporary_path = checkpoint_path.with_suffix('.tmp')
    torch.save(state, temporary_path)
    temporary_path.replace(checkpoint_path)
    epoch_path = Path(CHECKPOINT_DIR) / f'epoch_{next_epoch:04d}_step_{global_step:08d}.pt'
    torch.save(state, epoch_path)
    print(f'Saved checkpoint: {checkpoint_path}')


def move_targets_to_device(batch):
    target_boxes = [boxes.to(DEVICE) for boxes in batch['boxes']]
    target_labels = [labels.to(DEVICE) for labels in batch['labels']]
    return target_boxes, target_labels


def evaluate_loss(model, data_loader):
    """Compute mean DETR loss over a labeled evaluation split."""
    was_training = model.training
    model.eval()
    total_loss = 0.0
    batch_count = 0
    with torch.no_grad():
        for batch in data_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            target_boxes, target_labels = move_targets_to_device(batch)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                decoder_output, _ = model(images, masks=None)
                _, pred_boxes, pred_logits = decoder_output
                loss = criterion(pred_logits, pred_boxes, target_labels, target_boxes)
            total_loss += loss.float().item()
            batch_count += 1
    if was_training:
        model.train()
    return total_loss / max(1, batch_count)


def log_predictions_to_tensorboard(model, batch, writer, global_step):
    """Log denormalized images with ground-truth and predicted boxes."""
    was_training = model.training
    model.eval()
    images = batch['image'].to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            decoder_output, _ = model(images, masks=None)
            _, pred_boxes, pred_logits = decoder_output
        probabilities = pred_logits.float().softmax(dim=-1)
        pred_scores, pred_labels = probabilities[..., :-1].max(dim=-1)

    mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(3, 1, 1)
    scale = torch.tensor(
        [TARGET_SIZE[0], TARGET_SIZE[1], TARGET_SIZE[0], TARGET_SIZE[1]],
        device=DEVICE,
        dtype=pred_boxes.dtype,
    )

    for image_index in range(min(images.shape[0], MAX_PREDICTION_IMAGES)):
        image = (images[image_index] * std + mean).clamp(0, 1).cpu().permute(1, 2, 0).numpy()
        figure, axis = plt.subplots(figsize=(8, 8))
        axis.imshow(image)
        axis.set_axis_off()

        gt_boxes = cxcywh_to_xyxy(batch['boxes'][image_index]).cpu().numpy()
        gt_labels = batch['labels'][image_index].cpu().numpy()
        for box, label in zip(gt_boxes, gt_labels):
            x_min, y_min, x_max, y_max = box
            axis.add_patch(Rectangle(
                (x_min, y_min), x_max - x_min, y_max - y_min,
                fill=False, edgecolor='lime', linewidth=1.5,
            ))
            axis.text(x_min, y_min, f'GT {int(label)}', color='lime', fontsize=8,
                      bbox={'facecolor': 'black', 'alpha': 0.6, 'pad': 1})

        predicted_boxes = cxcywh_to_xyxy(pred_boxes[image_index]) * scale
        keep = pred_scores[image_index] >= PREDICTION_SCORE_THRESHOLD
        for box, label, score in zip(
            predicted_boxes[keep].cpu().numpy(),
            pred_labels[image_index][keep].cpu().numpy(),
            pred_scores[image_index][keep].cpu().numpy(),
        ):
            x_min, y_min, x_max, y_max = box
            axis.add_patch(Rectangle(
                (x_min, y_min), x_max - x_min, y_max - y_min,
                fill=False, edgecolor='red', linewidth=1.5,
            ))
            axis.text(x_min, y_max, f'Pred {int(label)} {score:.2f}', color='red', fontsize=8,
                      bbox={'facecolor': 'white', 'alpha': 0.7, 'pad': 1})

        axis.set_xlim(0, TARGET_SIZE[0])
        axis.set_ylim(TARGET_SIZE[1], 0)
        axis.set_title(f'Image {image_index}: green=GT, red=prediction')
        writer.add_figure('predictions', figure, global_step)
        plt.close(figure)

    if was_training:
        model.train()
    del images, pred_boxes, pred_logits, probabilities
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
model.train()
for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_loss = 0.0
    for batch_index, batch in enumerate(train_loader):
        if epoch == start_epoch and batch_index < start_batch:
            continue

        images = batch['image'].to(DEVICE, non_blocking=True)
        target_boxes, target_labels = move_targets_to_device(batch)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            decoder_output, _ = model(images, masks=None)
            _, pred_boxes, pred_logits = decoder_output
            loss = criterion(pred_logits, pred_boxes, target_labels, target_boxes)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = loss.detach().item()
        epoch_loss += loss_value
        global_step += 1
        writer.add_scalar('train/loss', loss_value, global_step)
        writer.add_scalar('train/learning_rate', optimizer.param_groups[0]['lr'], global_step)

        if global_step % 10 == 0:
            print(f'Epoch {epoch + 1}/{NUM_EPOCHS} | batch {batch_index + 1}/{len(train_loader)} | loss {loss_value:.4f}')

        next_epoch = epoch
        next_batch = batch_index + 1
        if next_batch >= len(train_loader):
            next_epoch = epoch + 1
            next_batch = 0
        if CHECKPOINT_DIR and global_step % CHECKPOINT_EVERY_STEPS == 0:
            save_checkpoint(next_epoch, next_batch, global_step)

        if global_step % PREDICTIONS_EVERY_STEPS == 0:
            log_predictions_to_tensorboard(model, batch, writer, global_step)

        del images, target_boxes, target_labels, loss
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    start_batch = 0
    mean_train_loss = epoch_loss / max(1, len(train_loader))
    mean_val_loss = evaluate_loss(model, val_loader)
    writer.add_scalar('train/epoch_loss', mean_train_loss, epoch + 1)
    writer.add_scalar('validation/loss', mean_val_loss, epoch + 1)
    save_checkpoint(epoch + 1, 0, global_step)
    print(f'Finished epoch {epoch + 1}; train loss: {mean_train_loss:.4f}; validation loss: {mean_val_loss:.4f}')

print('Training finished. Run the next cell for final validation KPIs.')

In [ ]:
print('Evaluating final model on the validation split...')
validation_metrics = evaluate_detr_model(
    model=model,
    data_loader=val_loader,
    device=DEVICE,
    image_width=TARGET_SIZE[0],
    image_height=TARGET_SIZE[1],
    num_classes=NUM_CLASSES,
    score_threshold=PREDICTION_SCORE_THRESHOLD,
    iou_threshold=0.5,
    use_amp=USE_AMP,
)

for metric_name, metric_value in validation_metrics.items():
    print(f'{metric_name}: {metric_value:.4f}')
    writer.add_scalar(f'validation/{metric_name}', metric_value, global_step)
writer.flush()
writer.close()

## TensorBoard

Run this cell after the training cell, or while training is running in another cell. If checkpoints are enabled, logs are stored beside them on Drive.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $log_dir